In [0]:
from pyspark.sql import functions as F

orders_silver = spark.table(
    "workspace.fashion_silver.orders"
)

order_items_silver = spark.table(
    "workspace.fashion_silver.order_items"
)

products_silver = spark.table(
    "workspace.fashion_silver.products"
)

customers_silver = spark.table(
    "workspace.fashion_silver.customers"
)

print("Sales analytics source tables loaded successfully.")

Sales analytics source tables loaded successfully.


In [0]:
orders_silver.groupBy("order_status") \
    .agg(
        F.count("*").alias("total_orders"),
        F.round(F.sum("net_order_amount"), 2).alias("net_order_value")
    ) \
    .orderBy(F.desc("total_orders")) \
    .show(truncate=False)

+------------+------------+---------------+
|order_status|total_orders|net_order_value|
+------------+------------+---------------+
|Delivered   |166872      |2.44941884387E9|
|Cancelled   |19930       |2.918766168E8  |
|Returned    |12028       |1.7637810814E8 |
|Processing  |810         |1.234186071E7  |
|Shipped     |360         |5285449.18     |
+------------+------------+---------------+



In [0]:
daily_sales = (
    orders_silver
    .filter(F.col("order_status") == "Delivered")
    .groupBy("order_date")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("total_items").alias("units_sold"),
        F.round(F.sum("gross_order_amount"), 2).alias("gross_revenue"),
        F.round(F.sum("discount_order_amount"), 2).alias("total_discounts"),
        F.round(F.sum("net_order_amount"), 2).alias("net_revenue"),
        F.round(F.avg("net_order_amount"), 2).alias("average_order_value")
    )
    .orderBy("order_date")
)

daily_sales.show(10, truncate=False)

+----------+------------+----------+-------------+---------------+-----------+-------------------+
|order_date|total_orders|units_sold|gross_revenue|total_discounts|net_revenue|average_order_value|
+----------+------------+----------+-------------+---------------+-----------+-------------------+
|2023-01-01|129         |588       |1821953.56   |107115.8       |1714837.76 |13293.32           |
|2023-01-02|107         |510       |1501272.34   |97350.12       |1403922.22 |13120.77           |
|2023-01-03|100         |481       |1543228.17   |106104.76      |1437123.41 |14371.23           |
|2023-01-04|114         |519       |1720413.88   |94574.32       |1625839.56 |14261.75           |
|2023-01-05|97          |534       |1737747.22   |100985.29      |1636761.93 |16873.83           |
|2023-01-06|129         |588       |2021576.31   |114905.72      |1906670.59 |14780.39           |
|2023-01-07|121         |600       |1845904.08   |99185.56       |1746718.52 |14435.69           |
|2023-01-0

In [0]:
daily_sales.agg(
    F.sum("total_orders").alias("total_delivered_orders"),
    F.sum("units_sold").alias("total_units_sold"),
    F.round(F.sum("gross_revenue"), 2).alias("total_gross_revenue"),
    F.round(F.sum("total_discounts"), 2).alias("total_discounts"),
    F.round(F.sum("net_revenue"), 2).alias("total_net_revenue"),
    F.count("*").alias("number_of_days")
).show(truncate=False)

+----------------------+----------------+-------------------+---------------+-----------------+--------------+
|total_delivered_orders|total_units_sold|total_gross_revenue|total_discounts|total_net_revenue|number_of_days|
+----------------------+----------------+-------------------+---------------+-----------------+--------------+
|166872                |833255          |2.60957705797E9    |1.601582141E8  |2.44941884387E9  |1392          |
+----------------------+----------------+-------------------+---------------+-----------------+--------------+



In [0]:
daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.fashion_gold.daily_sales_performance")

print("Gold table saved successfully.")

Gold table saved successfully.


In [0]:
gold_sales = spark.table(
    "workspace.fashion_gold.daily_sales_performance"
)

print("Total records:", gold_sales.count())
gold_sales.printSchema()
gold_sales.show(5, truncate=False)

Total records: 1392
root
 |-- order_date: date (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- units_sold: long (nullable = true)
 |-- gross_revenue: double (nullable = true)
 |-- total_discounts: double (nullable = true)
 |-- net_revenue: double (nullable = true)
 |-- average_order_value: double (nullable = true)

+----------+------------+----------+-------------+---------------+-----------+-------------------+
|order_date|total_orders|units_sold|gross_revenue|total_discounts|net_revenue|average_order_value|
+----------+------------+----------+-------------+---------------+-----------+-------------------+
|2023-01-01|129         |588       |1821953.56   |107115.8       |1714837.76 |13293.32           |
|2023-01-02|107         |510       |1501272.34   |97350.12       |1403922.22 |13120.77           |
|2023-01-03|100         |481       |1543228.17   |106104.76      |1437123.41 |14371.23           |
|2023-01-04|114         |519       |1720413.88   |94574.32       |162583